# Домен: Якість повітря (Beijing PM2.5)
Цільова змінна: концентрація PM2.5 (мкг/м3). Горизонт прогнозу: 1 година.
Метрики: MAE та RMSE.

## 1. Завантаження даних та EDA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
import os

# 1. Пошук файлу в середовищі Kaggle
file_path = ""
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if 'pm2' in filename.lower() and filename.endswith('.csv'):
            file_path = os.path.join(dirname, filename)
            break

if not file_path:
    print("Помилка: Файл не знайдено. Переконайтеся, що ви додали датасет через 'Add Data'.")
else:
    print(f"Файл знайдено за шляхом: {file_path}")
    
    # 2. Завантаження даних
    df_raw = pd.read_csv(file_path)
    
    # 3. Формування DatetimeIndex з окремих колонок
    df_raw['datetime'] = pd.to_datetime(
        df_raw[['year', 'month', 'day', 'hour']]
    )
    df_raw.set_index('datetime', inplace=True)
    
    # 4. Видалення службових колонок
    df_raw.drop(['No', 'year', 'month', 'day', 'hour'], axis=1, inplace=True)
    
    # 5. Обробка пропусків у pm2.5 (лінійна інтерполяція)
    missing_count = df_raw['pm2.5'].isna().sum()
    print(f"Пропусків у pm2.5: {missing_count} ({missing_count / len(df_raw) * 100:.1f}%)")
    df_raw['pm2.5'] = df_raw['pm2.5'].interpolate(method='linear')
    df_raw['pm2.5'] = df_raw['pm2.5'].bfill()  # Заповнюємо початкові NaN зворотним заповненням
    
    # 6. One-hot encoding для напрямку вітру (cbwd)
    df_pm = pd.get_dummies(df_raw, columns=['cbwd'], prefix='wind', dtype=int)
    
    target_col = 'pm2.5'
    
    print(f"Дані успішно підготовлено! Кількість годин: {len(df_pm)}")
    print(f"Колонки: {list(df_pm.columns)}")
    print(f"\nПерші 3 рядки:")
    print(df_pm.head(3))

In [ ]:
# Візуалізація ряду PM2.5
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# 1. Часовий ряд
axes[0].plot(df_pm.index, df_pm['pm2.5'], color='purple', linewidth=0.3, alpha=0.8)
axes[0].set_title('Beijing PM2.5: Погодинна концентрація')
axes[0].set_ylabel('PM2.5 (мкг/м3)')
axes[0].grid(True, alpha=0.3)

# 2. Гістограма розподілу
axes[1].hist(df_pm['pm2.5'], bins=100, color='purple', alpha=0.7, edgecolor='black', linewidth=0.3)
axes[1].set_title('Розподіл PM2.5 (зміщення вправо характерне для забрудненого повітря)')
axes[1].set_xlabel('PM2.5 (мкг/м3)')
axes[1].set_ylabel('Частота')
axes[1].grid(True, alpha=0.3)

# 3. Boxplot по місяцях (сезонність)
df_pm['month_temp'] = df_pm.index.month
df_pm.boxplot(column='pm2.5', by='month_temp', ax=axes[2])
axes[2].set_title('Сезонність PM2.5 по місяцях (опалювальний сезон = зима)')
axes[2].set_xlabel('Місяць')
axes[2].set_ylabel('PM2.5 (мкг/м3)')
df_pm.drop('month_temp', axis=1, inplace=True)
plt.suptitle('')

plt.tight_layout()
plt.show()

# Тест ADF на стаціонарність
def check_stationarity(series, name):
    print(f"\n--- Тест ADF для: {name} ---")
    result = adfuller(series)
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4e}')
    if result[1] <= 0.05:
        print("Висновок: Ряд стаціонарний (H0 відхилено). Ряд не потребує диференціювання.")
    else:
        print("Висновок: Ряд НЕ стаціонарний (H0 не відхилено).")

check_stationarity(df_pm['pm2.5'], "Концентрація PM2.5")

# Базова статистика
print("\n--- Базова статистика ---")
print(df_pm.describe().T[['count', 'mean', 'min', 'max', 'std']])

## 2. Підготовка даних та Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

print("Підготовка даних для моделей прогнозування PM2.5...")

# 1. Вибір ознак для моделі
# Екзогенні змінні: DEWP, TEMP, PRES, Iws, Is, Ir + one-hot wind
target_col = 'pm2.5'
feature_cols = [col for col in df_pm.columns]  # Усі колонки

# 2. Lookback window: 24 години (добовий цикл)
lookback = 24
df_ml = pd.DataFrame(index=df_pm.index)

# Цільова змінна (прогнозуємо PM2.5 на час t)
df_ml['target'] = df_pm[target_col]

# Лаги для КОЖНОЇ ознаки (24 години назад)
for col in feature_cols:
    for i in range(1, lookback + 1):
        df_ml[f'{col}_lag_{i}'] = df_pm[col].shift(i)

# Календарні ознаки (фіксуємо добовий та сезонний цикл)
df_ml['hour'] = df_ml.index.hour
df_ml['month'] = df_ml.index.month

# Видаляємо рядки з NaN після .shift()
df_ml.dropna(inplace=True)

# 3. Формування X та y
X_pm = df_ml.drop(['target'], axis=1).values
y_pm = df_ml['target'].values

# 4. Train/Test split: тест = останні 30 днів (720 годин)
test_size = 720
train_idx = len(df_ml) - test_size

X_train_pm, X_test_pm = X_pm[:train_idx], X_pm[train_idx:]
y_train_pm, y_test_pm = y_pm[:train_idx], y_pm[train_idx:]

# 5. Масштабування
scaler_x_pm = StandardScaler()
scaler_y_pm = StandardScaler()

X_train_pm_s = scaler_x_pm.fit_transform(X_train_pm)
X_test_pm_s = scaler_x_pm.transform(X_test_pm)

y_train_pm_s = scaler_y_pm.fit_transform(y_train_pm.reshape(-1, 1))
y_test_pm_s = scaler_y_pm.transform(y_test_pm.reshape(-1, 1))

# 6. 3D-тензори для LSTM та 1D-CNN
X_train_pm_3d = X_train_pm_s.reshape((X_train_pm_s.shape[0], 1, X_train_pm_s.shape[1]))
X_test_pm_3d = X_test_pm_s.reshape((X_test_pm_s.shape[0], 1, X_test_pm_s.shape[1]))

print("Підготовку даних завершено!")
print(f"Кількість ознак: {X_train_pm_s.shape[1]}")
print(f"Розмір тренувальної вибірки: {X_train_pm_s.shape[0]} годин")
print(f"Розмір тестової вибірки: {X_test_pm_s.shape[0]} годин")

## 3. Базові моделі (Baselines)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Тестова вибірка з оригінального ряду для baseline
test_size = 720
test_data_pm = df_pm.iloc[-test_size:].copy()
true_values_pm = test_data_pm[target_col].values

# 1. Naive Forecast: PM2.5 наступної години = PM2.5 цієї години (t-1)
test_data_pm['naive_pred'] = df_pm[target_col].shift(1).iloc[-test_size:].values

# 2. Seasonal Naive: PM2.5 = значення 24 години тому (t-24)
test_data_pm['s_naive_pred'] = df_pm[target_col].shift(24).iloc[-test_size:].values

# Функція для розрахунку MAE та RMSE
def print_pm_metrics(name, true, pred):
    mae = mean_absolute_error(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    print(f"{name:<15} | MAE: {mae:6.2f} мкг/м3 | RMSE: {rmse:6.2f} мкг/м3")

print("--- БАЗОВI МОДЕЛI (PM2.5) ---")
print_pm_metrics("Naive (t-1)", true_values_pm, test_data_pm['naive_pred'])
print_pm_metrics("S. Naive (t-24)", true_values_pm, test_data_pm['s_naive_pred'])

## 4. Етап 1: Моделі з базовими конфігураціями
Усі моделі навчаються з мінімально необхідними параметрами для отримання baseline результатів.

### 4.1. XGBoost

In [ ]:
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

print("Навчання XGBoost (PM2.5)...")

model_xgb_pm = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

# Для дерев y не потребує масштабування
model_xgb_pm.fit(X_train_pm_s, y_train_pm)
xgb_pred_pm = model_xgb_pm.predict(X_test_pm_s)

mae_xgb = mean_absolute_error(y_test_pm, xgb_pred_pm)
rmse_xgb = np.sqrt(mean_squared_error(y_test_pm, xgb_pred_pm))

print(f"\nXGBoost      | MAE: {mae_xgb:6.2f} мкг/м3 | RMSE: {rmse_xgb:6.2f} мкг/м3")

### 4.2. SVR (Support Vector Regression)

In [ ]:
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

print("Навчання SVR (PM2.5). Використовується сабсемплінг для O(n^3) складності...")

# Обмежуємо тренувальну вибірку для SVR
train_subset = 10000

model_svr_pm = SVR(kernel='rbf', C=1.0, epsilon=0.1, gamma='scale')
model_svr_pm.fit(X_train_pm_s[-train_subset:], y_train_pm[-train_subset:])
svr_pred_pm = model_svr_pm.predict(X_test_pm_s)

mae_svr = mean_absolute_error(y_test_pm, svr_pred_pm)
rmse_svr = np.sqrt(mean_squared_error(y_test_pm, svr_pred_pm))

print(f"\nSVR          | MAE: {mae_svr:6.2f} мкг/м3 | RMSE: {rmse_svr:6.2f} мкг/м3")

### 4.3. Нейронні мережі (MLP, LSTM, 1D-CNN)

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Conv1D, Flatten, Input
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

tf.keras.utils.set_random_seed(42)

print("Навчання нейромереж (PM2.5). Зачекайте 1-2 хвилини...\n")

# 1. MLP
model_mlp_pm = Sequential([
    Input(shape=(X_train_pm_s.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)
])
model_mlp_pm.compile(optimizer='adam', loss='mse')
model_mlp_pm.fit(X_train_pm_s, y_train_pm_s, epochs=50, batch_size=256, validation_split=0.1, verbose=0)
mlp_pred_pm_s = model_mlp_pm.predict(X_test_pm_s, verbose=0)
mlp_pred_pm = scaler_y_pm.inverse_transform(mlp_pred_pm_s).flatten()

# 2. LSTM
model_lstm_pm = Sequential([
    Input(shape=(1, X_train_pm_s.shape[1])),
    LSTM(64, activation='relu', return_sequences=False),
    Dense(16, activation='relu'),
    Dense(1)
])
model_lstm_pm.compile(optimizer='adam', loss='mse')
model_lstm_pm.fit(X_train_pm_3d, y_train_pm_s, epochs=50, batch_size=256, validation_split=0.1, verbose=0)
lstm_pred_pm_s = model_lstm_pm.predict(X_test_pm_3d, verbose=0)
lstm_pred_pm = scaler_y_pm.inverse_transform(lstm_pred_pm_s).flatten()

# 3. 1D-CNN
model_cnn_pm = Sequential([
    Input(shape=(1, X_train_pm_s.shape[1])),
    Conv1D(filters=64, kernel_size=1, activation='relu'),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(1)
])
model_cnn_pm.compile(optimizer='adam', loss='mse')
model_cnn_pm.fit(X_train_pm_3d, y_train_pm_s, epochs=50, batch_size=256, validation_split=0.1, verbose=0)
cnn_pred_pm_s = model_cnn_pm.predict(X_test_pm_3d, verbose=0)
cnn_pred_pm = scaler_y_pm.inverse_transform(cnn_pred_pm_s).flatten()

# Результати
def print_dl_pm(name, pred):
    mae = mean_absolute_error(y_test_pm, pred)
    rmse = np.sqrt(mean_squared_error(y_test_pm, pred))
    print(f"{name:<12} | MAE: {mae:6.2f} мкг/м3 | RMSE: {rmse:6.2f} мкг/м3")

print("--- РЕЗУЛЬТАТИ DEEP LEARNING (PM2.5) ---")
print_dl_pm("MLP", mlp_pred_pm)
print_dl_pm("LSTM", lstm_pred_pm)
print_dl_pm("1D-CNN", cnn_pred_pm)

### 4.4. Класичні статистичні моделі (ARIMA, SARIMA, Holt-Winters)

In [ ]:
import numpy as np
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings

warnings.filterwarnings('ignore')

print("Навчання класичних статистичних моделей (PM2.5)...\n")

# Для статистики використовуємо одновимірний ряд
train_stat_pm = y_train_pm[-2000:]
test_stat_pm = y_test_pm

# 1. ARIMA (без сезонного компонента)
print("1. Навчання ARIMA(3,0,3)...")
model_arima_pm = SARIMAX(train_stat_pm, order=(3, 0, 3), trend='c')
res_arima_pm = model_arima_pm.fit(disp=False)
pred_arima_pm = res_arima_pm.apply(test_stat_pm).fittedvalues

# 2. SARIMA (з добовою сезонністю 24 години)
print("2. Навчання SARIMA(1,0,1)(1,0,1,24)... (зачекайте близько хвилини)")
model_sarima_pm = SARIMAX(train_stat_pm, order=(1, 0, 1), seasonal_order=(1, 0, 1, 24), trend='c')
res_sarima_pm = model_sarima_pm.fit(disp=False)
pred_sarima_pm = res_sarima_pm.apply(test_stat_pm).fittedvalues

# 3. Holt-Winters (Rolling forecast)
print("3. Прогноз Holt-Winters (Rolling forecast)...")
hw_preds_pm = []
history_hw_pm = list(train_stat_pm)

for t in range(len(test_stat_pm)):
    window = history_hw_pm[-120:]  # Останні 120 годин (5 діб)
    model_hw_pm = ExponentialSmoothing(
        window, seasonal='add', seasonal_periods=24, trend=None
    ).fit(optimized=True)
    pred = model_hw_pm.forecast(1)[0]
    hw_preds_pm.append(pred)
    history_hw_pm.append(test_stat_pm[t])

pred_hw_pm = np.array(hw_preds_pm)

# Результати
def print_stat_pm(name, pred):
    mae = mean_absolute_error(test_stat_pm, pred)
    rmse = np.sqrt(mean_squared_error(test_stat_pm, pred))
    print(f"{name:<15} | MAE: {mae:6.2f} мкг/м3 | RMSE: {rmse:6.2f} мкг/м3")

print("\n--- РЕЗУЛЬТАТИ СТАТИСТИКИ (PM2.5, 1 крок вперед) ---")
print_stat_pm("ARIMA(3,0,3)", pred_arima_pm)
print_stat_pm("SARIMA(24h)", pred_sarima_pm)
print_stat_pm("Holt-Winters", pred_hw_pm)

## 5. Етап 2: Оптимізація та експерименти
Покращені версії моделей з додатковими техніками регуляризації та оптимізації гіперпараметрів.
Буде реалізовано за явним запитом.

## 6. Фінальна зведена таблиця

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("--- ФІНАЛЬНА ЗВЕДЕНА ТАБЛИЦЯ (PM2.5, Beijing) ---")

results = []

def calc_pm_metrics(name, cls, true, pred):
    mae = mean_absolute_error(true, pred)
    rmse = np.sqrt(mean_squared_error(true, pred))
    return {"Модель": name, "Клас": cls, "MAE (мкг/м3)": round(mae, 2), "RMSE (мкг/м3)": round(rmse, 2)}

# Baselines
results.append(calc_pm_metrics("Naive (t-1)", "Baseline", true_values_pm, test_data_pm['naive_pred']))
results.append(calc_pm_metrics("S. Naive (t-24)", "Baseline", true_values_pm, test_data_pm['s_naive_pred']))

# ML
results.append(calc_pm_metrics("XGBoost", "Traditional ML", y_test_pm, xgb_pred_pm))
results.append(calc_pm_metrics("SVR", "Traditional ML", y_test_pm, svr_pred_pm))

# DL
results.append(calc_pm_metrics("MLP", "Deep Learning", y_test_pm, mlp_pred_pm))
results.append(calc_pm_metrics("LSTM", "Deep Learning", y_test_pm, lstm_pred_pm))
results.append(calc_pm_metrics("1D-CNN", "Deep Learning", y_test_pm, cnn_pred_pm))

# Statistics
results.append(calc_pm_metrics("ARIMA(3,0,3)", "Classic Statistics", test_stat_pm, pred_arima_pm))
results.append(calc_pm_metrics("SARIMA(24h)", "Classic Statistics", test_stat_pm, pred_sarima_pm))
results.append(calc_pm_metrics("Holt-Winters", "Classic Statistics", test_stat_pm, pred_hw_pm))

# Сортуємо за MAE
results_df = pd.DataFrame(results).sort_values(by="MAE (мкг/м3)").reset_index(drop=True)
results_df.index = results_df.index + 1
print(results_df.to_string())